# LeetCode 3963 — Create Grid With Exactly One Path bằng Python

Link đề: https://leetcode.com/problems/create-grid-with-exactly-one-path/ · Độ khó: **Easy** · Chủ đề: Array, Matrix, Construction, Dynamic Programming

Skeleton LeetCode:

```python
class Solution:
    def createGrid(self, m: int, n: int) -> List[str]:
        ...
```

> Ghi chú: tên bài, ví dụ, constraints và chữ ký hàm được đối chiếu qua leetcode.doocs.org (Easy, không Premium); phần phát biểu đề được viết lại bằng lời của mình. Slug URL được suy ra từ tên bài.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề **xây dựng (construction)**: tạo **một** lưới `m x n` gồm `'.'` (ô trống) và `'#'` (chướng ngại) sao cho có **đúng một** đường đi từ góc trên trái đến góc dưới phải, chỉ đi **sang phải hoặc xuống**.
2. Biết **kiểm chứng** đáp án bằng quy hoạch động đếm số đường đi (`dp[i][j] = dp[i-1][j] + dp[i][j-1]` trên ô trống).
3. Nhận ra bước đột phá: nếu các ô trống tạo thành **một dải liền mạch duy nhất** đi từ đầu đến cuối thì chắc chắn chỉ có một đường đi; hình chữ L (hàng đầu + cột cuối) là cách đơn giản nhất.
4. Viết 4 cách dựng (chữ L trên-phải, chữ L trái-dưới, bậc thang, dựng theo mô phỏng đường đi) và kiểm tra tất cả bằng **bộ đếm đường đi**, kèm vét cạn các lưới nhỏ.
5. Tránh lỗi: để lưới toàn `.` (nhiều đường), chỉ mở hàng đầu (không tới đích), mở cả hàng đầu lẫn hàng cuối (hai đường), nhầm kích thước `m` hàng `n` cột.

## 2. Đề bài & ví dụ

Cho hai số nguyên `m` và `n`. Hãy trả về **một** lưới `m x n` (list gồm `m` chuỗi, mỗi chuỗi dài `n`) chỉ chứa:

- `'.'`: ô trống (đi được),
- `'#'`: chướng ngại (không đi được),

sao cho có **đúng một** đường đi hợp lệ từ ô trên trái `(0, 0)` đến ô dưới phải `(m-1, n-1)`. Đường đi **chỉ** di chuyển sang **phải** hoặc **xuống**, chỉ qua ô `'.'` (kể cả ô đầu và ô cuối). Đề cho phép trả về **bất kỳ** lưới hợp lệ.

| Ví dụ | `m, n` | Output (một đáp án hợp lệ) | Đường đi duy nhất |
|---|---|---|---|
| 1 | `2, 3` | `["..#", "#.."]` | `(0,0)->(0,1)->(1,1)->(1,2)` |
| 2 | `3, 3` | `["..#", "#..", "##."]` | `(0,0)->(0,1)->(1,1)->(1,2)->(2,2)` |
| 3 | `1, 4` | `["...."]` | cả hàng, `(0,0)->...->(0,3)` |

Constraints: `1 <= m, n <= 25`.

Vì có nhiều đáp án đúng nên LeetCode chấm bằng **bộ kiểm tra** (đếm số đường đi). Ta cũng sẽ viết bộ kiểm tra như vậy.

In [1]:
for m, n, grid in [(2, 3, ["..#", "#.."]), (3, 3, ["..#", "#..", "##."]), (1, 4, ["...."])]:
    print(f"m={m}, n={n}")
    for row in grid:
        print("   ", row)

m=2, n=3
    ..#
    #..
m=3, n=3
    ..#
    #..
    ##.
m=1, n=4
    ....


## 3. Kiến thức nền

### 3.1. Lưới là list các chuỗi

`grid = ["..#", "#.."]` có `m = 2` hàng, mỗi hàng là chuỗi độ dài `n = 3`. `grid[i][j]` là ký tự ở hàng `i`, cột `j`. Chuỗi **bất biến** (không gán `grid[i][j] = ...` được), nên khi dựng ta dùng list các list ký tự rồi `"".join(row)`.

In [2]:
grid = ["..#", "#.."]
print(len(grid), len(grid[0]), grid[0][1], grid[1][0])
try:
    grid[0][0] = "#"
except TypeError as e:
    print("TypeError:", e)

2 3 . #
TypeError: 'str' object does not support item assignment


### 3.2. Dựng lưới bằng list lồng và `"".join`

`[["#"] * n for _ in range(m)]` tạo lưới `m x n` toàn `#`, với **mỗi hàng là một list riêng**. Sau khi sửa ô, đổi từng hàng thành chuỗi bằng `"".join(row)`.

**Cạm bẫy:** `[["#"] * n] * m` tạo `m` tham chiếu tới **cùng một** list hàng; sửa một hàng sẽ sửa tất cả.

In [3]:
g = [["#"] * 3 for _ in range(2)]
g[0][0] = "."
print(["".join(r) for r in g])
bad = [["#"] * 3] * 2
bad[0][0] = "."
print(["".join(r) for r in bad])             # cả hai hàng cùng đổi!

['.##', '###']
['.##', '.##']


### 3.3. Đếm số đường đi bằng quy hoạch động (DP)

`dp[i][j]` = số đường đi (chỉ phải/xuống) từ `(0, 0)` đến `(i, j)`. Nếu ô là `#` thì `dp = 0`. Với ô `.`: `dp[i][j] = dp[i-1][j] + dp[i][j-1]` (nguồn từ trên hoặc từ trái), và `dp[0][0] = 1` nếu ô đầu trống.

In [4]:
def count_paths(grid):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if grid[i][j] == "#":
                continue
            if i == 0 and j == 0:
                dp[i][j] = 1
            else:
                dp[i][j] = (dp[i - 1][j] if i else 0) + (dp[i][j - 1] if j else 0)
    return dp[m - 1][n - 1]

print(count_paths(["..#", "#.."]), count_paths(["...", "..."]), count_paths(["#.."]))

1 3 0


### 3.4. Kiểm tra định dạng đầu ra

Trước khi đếm đường, kiểm tra: đúng `m` chuỗi, mỗi chuỗi dài `n`, chỉ chứa `.` và `#`. Một hàm `valid(grid, m, n)` gom cả hai bước: nó là **trọng tài** của notebook này.

In [5]:
def valid(grid, m, n):
    if len(grid) != m or any(len(r) != n for r in grid):
        return False
    if any(c not in ".#" for r in grid for c in r):
        return False
    return count_paths(grid) == 1

print(valid(["..#", "#.."], 2, 3), valid(["...", "..."], 2, 3), valid(["..#", "#.."], 3, 3))

True False False


### 3.5. Vét cạn các lưới nhỏ bằng `itertools.product`

Với `m * n` nhỏ, ta liệt kê mọi lưới `2^(m*n)` bằng `product(".#", repeat=m*n)` để thấy bài toán có **rất nhiều** đáp án hợp lệ, và kiểm chứng công thức.

In [6]:
from itertools import product

def all_valid(m, n):
    res = []
    for cells in product(".#", repeat=m * n):
        grid = ["".join(cells[i * n:(i + 1) * n]) for i in range(m)]
        if count_paths(grid) == 1:
            res.append(grid)
    return res

print(len(all_valid(2, 2)), len(all_valid(2, 3)), len(all_valid(3, 3)))

2 5 26


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đề không bắt ta *tìm* đường mà **tự thiết kế lưới** để đường duy nhất tồn tại. Hãy chọn trước một **đường đi** `P` đi từ `(0, 0)` đến `(m-1, n-1)` bằng các bước phải/xuống, rồi **chỉ mở** các ô của `P` (đặt `'.'`), tất cả ô khác là `'#'`.

Khi đó mọi đường đi hợp lệ phải đi qua ô trống, tức là phải nằm hoàn toàn trong `P`. Mà `P` là một dải mỏng (mỗi ô có đúng một ô kế tiếp trong `P`), nên chỉ có **một** cách đi từ đầu đến cuối: đi theo `P`.

Ví dụ `m = 2, n = 3` với `P` = "phải, phải, xuống": `(0,0),(0,1),(0,2),(1,2)`:

```
...
##.
```

Chỉ một đường đi. Ví dụ của đề dùng `P` khác (bậc thang) nhưng cùng nguyên tắc.

In [7]:
g = ["...", "##."]
print(count_paths(g), valid(g, 2, 3))
g2 = ["..#", "#.."]
print(count_paths(g2), valid(g2, 2, 3))

1 True
1 True


## 5. Cách 1 — Brute Force / baseline: vét cạn lưới nhỏ (chỉ minh họa)

Duyệt mọi lưới (`2^(m*n)`), trả lưới đầu tiên có đúng một đường đi. Đúng nhưng chỉ chạy được khi `m * n <= ~20`; với `25 x 25 = 625` ô thì **không thể**. Đây là "nút thắt" khiến ta phải **dựng** thay vì **tìm**.

In [8]:
from itertools import product

class SolutionBrute:
    def createGrid(self, m, n):
        for cells in product(".#", repeat=m * n):
            grid = ["".join(cells[i * n:(i + 1) * n]) for i in range(m)]
            if count_paths(grid) == 1:
                return grid

for m, n in [(1, 4), (2, 3), (3, 3)]:
    g = SolutionBrute().createGrid(m, n)
    print(m, n, g, valid(g, m, n))

1 4 ['....'] True
2 3 ['...', '.#.'] True
3 3 ['...', '.#.', '.#.'] True


**Độ phức tạp:** Time `O(2^(m n) * m n)`, Space `O(m n)`. Chỉ dùng để thấy hình dáng nghiệm với lưới bé.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** chọn **một đường đi cụ thể** và mở đúng các ô của nó. Có nhiều lựa chọn, đơn giản nhất là **chữ L**: đi hết **hàng đầu** (sang phải) rồi đi hết **cột cuối** (xuống).

- Ô `(0, j)` với mọi `j` là `.`; ô `(i, n-1)` với mọi `i` là `.`; còn lại `#`.
- Đường duy nhất: `(0,0) -> (0,1) -> ... -> (0,n-1) -> (1,n-1) -> ... -> (m-1,n-1)`.

**Chứng minh nhanh số đường đi là 1:** ô `(i, j)` với `i >= 1` và `j < n-1` là `#`, nên đã vào hàng `i >= 1` thì chỉ còn cột `n-1` (không thể sang phải khi đã ở cột cuối, chỉ xuống). Còn ở hàng `0`, chỉ có thể sang phải tới cột `n-1` rồi mới xuống được (xuống ở cột `j < n-1` rơi vào `#`).

**Quy tắc:** *mở đúng một dải ô nối đầu với cuối theo thứ tự phải/xuống; mọi ô khác là chướng ngại.* Mọi dải như thế đều hợp lệ, vì mỗi ô trên dải chỉ có đúng một ô kế tiếp trong dải.

## 7. Các cách tiếp theo

### Cách 2 — Chữ L trên-phải (hàng đầu + cột cuối)

Dựng lưới `#`, mở hàng 0 và cột `n - 1`.

In [9]:
class SolutionL:
    def createGrid(self, m, n):
        g = [["#"] * n for _ in range(m)]
        g[0] = ["."] * n                    # mở cả hàng đầu
        for i in range(m):
            g[i][n - 1] = "."              # mở cả cột cuối
        return ["".join(row) for row in g]

for m, n in [(2, 3), (3, 3), (1, 4)]:
    print(m, n, SolutionL().createGrid(m, n))

2 3 ['...', '##.']
3 3 ['...', '##.', '##.']
1 4 ['....']


**Độ phức tạp:** Time `O(m n)` (dựng lưới), Space `O(m n)`.

### Cách 3 — Chữ L đối xứng (cột đầu + hàng cuối)

Đường đi: xuống hết cột 0 rồi sang phải hết hàng cuối. Cũng đúng; thể hiện tính **đa nghiệm**.

In [10]:
class SolutionL2:
    def createGrid(self, m, n):
        g = [["#"] * n for _ in range(m)]
        for i in range(m):
            g[i][0] = "."                  # mở cột đầu
        g[m - 1] = ["."] * n               # mở hàng cuối
        return ["".join(row) for row in g]

for m, n in [(2, 3), (3, 3), (1, 4)]:
    print(m, n, SolutionL2().createGrid(m, n))

2 3 ['.##', '...']
3 3 ['.##', '.##', '...']
1 4 ['....']


**Độ phức tạp:** Time `O(m n)`, Space `O(m n)`.

### Cách 4 — Bậc thang (khớp ví dụ của đề) bằng mô phỏng đường đi

Mô phỏng một con tốt đi từ `(0, 0)`: ở mỗi hàng, đi sang phải **một bước** (nếu còn cột) rồi xuống; đến hàng cuối thì đi sang phải hết. Mỗi ô con tốt đi qua được đặt `'.'`. Cách này dựng **chính đường đi** nên chứng minh đúng gần như tự nhiên, và ra đúng ví dụ của đề.

In [11]:
class SolutionStairs:
    def createGrid(self, m, n):
        g = [["#"] * n for _ in range(m)]
        i = j = 0
        g[0][0] = "."
        while (i, j) != (m - 1, n - 1):
            if i == m - 1:                  # hàng cuối: chỉ còn đi phải
                j += 1
            elif j == n - 1:                # cột cuối: chỉ còn đi xuống
                i += 1
            elif (i + j) % 2 == 0:          # xen kẽ: phải rồi xuống
                j += 1
            else:
                i += 1
            g[i][j] = "."
        return ["".join(row) for row in g]

for m, n in [(2, 3), (3, 3), (1, 4)]:
    print(m, n, SolutionStairs().createGrid(m, n))

2 3 ['..#', '#..']
3 3 ['..#', '#..', '##.']
1 4 ['....']


**Độ phức tạp:** Time `O(m n)` (dựng lưới; số bước mô phỏng `m + n - 2`), Space `O(m n)`.

## 8. TRACE từng bước

Trace Cách 4 với `m = 3, n = 3` (đường đi bậc thang):

| bước | vị trí `(i, j)` | quyết định | lưới sau bước |
|---|---|---|---|
| 0 | `(0,0)` | mở ô đầu | `.##`, `###`, `###` |
| 1 | `(0,1)` | `(0+0)%2==0` -> phải | `..#`, `###`, `###` |
| 2 | `(1,1)` | `(0+1)%2==1` -> xuống | `..#`, `#.#`, `###` |
| 3 | `(1,2)` | `(1+1)%2==0` -> phải | `..#`, `#..`, `###` |
| 4 | `(2,2)` | `j == n-1` -> xuống | `..#`, `#..`, `##.` |

Kết quả khớp ví dụ 2 của đề.

In [12]:
def trace_stairs(m, n):
    g = [["#"] * n for _ in range(m)]
    i = j = 0
    g[0][0] = "."
    print(f"bước 0: ({i},{j})", ["".join(r) for r in g])
    step = 0
    while (i, j) != (m - 1, n - 1):
        step += 1
        if i == m - 1:
            move, j = "phải (hàng cuối)", j + 1
        elif j == n - 1:
            move, i = "xuống (cột cuối)", i + 1
        elif (i + j) % 2 == 0:
            move, j = "phải", j + 1
        else:
            move, i = "xuống", i + 1
        g[i][j] = "."
        print(f"bước {step}: {move} -> ({i},{j})", ["".join(r) for r in g])
    return ["".join(r) for r in g]

assert trace_stairs(3, 3) == ["..#", "#..", "##."]
assert trace_stairs(2, 3) == ["..#", "#.."]

bước 0: (0,0) ['.##', '###', '###']
bước 1: phải -> (0,1) ['..#', '###', '###']
bước 2: xuống -> (1,1) ['..#', '#.#', '###']
bước 3: phải -> (1,2) ['..#', '#..', '###']
bước 4: xuống (cột cuối) -> (2,2) ['..#', '#..', '##.']
bước 0: (0,0) ['.##', '###']
bước 1: phải -> (0,1) ['..#', '###']
bước 2: xuống -> (1,1) ['..#', '#.#']
bước 3: phải (hàng cuối) -> (1,2) ['..#', '#..']


In [13]:
def trace_dp(grid):
    m, n = len(grid), len(grid[0])
    dp = [[0] * n for _ in range(m)]
    for i in range(m):
        for j in range(n):
            if grid[i][j] == "#":
                continue
            dp[i][j] = 1 if (i, j) == (0, 0) else (dp[i - 1][j] if i else 0) + (dp[i][j - 1] if j else 0)
    for row in dp:
        print("  ", row)
    print("  số đường đi:", dp[-1][-1])
    return dp[-1][-1]

assert trace_dp(["..#", "#..", "##."]) == 1
assert trace_dp(["...", "..."]) == 3

   [1, 1, 0]
   [0, 1, 1]
   [0, 0, 1]
  số đường đi: 1
   [1, 1, 1]
   [1, 2, 3]
  số đường đi: 3


## 9. Code nộp LeetCode

Dùng Cách 2 (chữ L trên-phải). Đề cho phép mọi lưới hợp lệ.

In [14]:
from typing import List, Optional


class Solution:
    def createGrid(self, m: int, n: int) -> List[str]:
        g = [["#"] * n for _ in range(m)]
        g[0] = ["."] * n
        for i in range(m):
            g[i][n - 1] = "."
        return ["".join(row) for row in g]


print(Solution().createGrid(2, 3), Solution().createGrid(3, 3), Solution().createGrid(1, 4))

['...', '##.'] ['...', '##.', '##.'] ['....']


## 10. Vì sao thuật toán đúng

Xét Cách 2 (chữ L trên-phải). Ô trống là `T = {(0, j) : 0 <= j < n} ∪ {(i, n-1) : 0 <= i < m}`.

1. **Có ít nhất một đường.** Đi sang phải dọc hàng 0 qua `(0,0), ..., (0, n-1)` rồi xuống dọc cột `n-1` tới `(m-1, n-1)`. Mọi ô trên đó thuộc `T`.
2. **Có nhiều nhất một đường.** Xét một đường hợp lệ bất kỳ. Nó chỉ đi phải/xuống nên chỉ số hàng và cột không giảm. Nếu tại ô `(i, j)` với `j < n-1`, bước xuống tới `(i+1, j)`: ô này có `i + 1 >= 1` và `j < n - 1` nên là `#` -> không hợp lệ. Vậy khi còn ở cột `j < n - 1` đường chỉ được sang phải; chỉ có thể xuống khi `j = n - 1`. Do đó đường phải đi sang phải đủ `n - 1` bước ở hàng 0 rồi xuống đủ `m - 1` bước ở cột cuối: duy nhất.
3. **Trường hợp biên.** `m = 1`: hàng 0 là cả lưới, một đường đi ngang. `n = 1`: cột `n - 1 = 0` là cả lưới, một đường đi dọc. `m = n = 1`: một ô `.`, đường đi độ dài 0 tính là một đường.
4. **Các cách khác** (chữ L đối xứng, bậc thang) tương tự: mở đúng một dải `P` mà mỗi ô có tối đa một ô kế tiếp trong `P` theo hướng phải/xuống, nên số đường đi là 1. Ta xác nhận lại bằng bộ đếm DP ở mục 12.

## 11. Phân tích Time / Space Complexity

- Cách 1 (vét cạn): Time `O(2^(mn) * mn)`, chỉ cho lưới cực nhỏ.
- Cách 2, 3 (chữ L): Time `O(m n)`, Space `O(m n)` (kích thước đầu ra, không thể ít hơn).
- Cách 4 (bậc thang): Time `O(m n)` để khởi tạo lưới + `O(m + n)` mô phỏng; Space `O(m n)`.
- Bộ kiểm tra `count_paths`: Time `O(m n)`, Space `O(m n)`.

Với `m, n <= 25` đầu ra tối đa `625` ký tự; mọi cách dựng chạy tức thì.

## 12. Bộ test

### Cách kiểm thử trong notebook này

Chúng ta dùng ba lớp kiểm thử, từ dễ đến khó:

1. **Test cố định**: ví dụ của đề + các trường hợp biên tự nghĩ ra. Mỗi test là một `assert hàm(input) == kết_quả_mong_đợi`. Nếu sai, Python ném `AssertionError` và dừng ngay, kèm thông tin ta in ra để dễ lần lại.
2. **Test ngẫu nhiên có `random.seed`**: gọi `random.seed(số)` trước vòng lặp để **mỗi lần chạy lại sinh đúng cùng dữ liệu** (tái lập được). Input luôn được sinh **trong đúng constraints** của đề.
3. **Đối chiếu nhiều cách (oracle)**: một cách rất đơn giản, hiển nhiên đúng (brute force) đóng vai "trọng tài"; mọi cách nhanh hơn phải cho cùng kết quả với trọng tài trên mọi input ngẫu nhiên.

Vì đề chấp nhận **mọi** lưới hợp lệ nên ta **không** so sánh với một đáp án cố định mà dùng hàm `valid(grid, m, n)` (đếm đường bằng DP) làm trọng tài.

In [15]:
import random

builders = [SolutionL(), SolutionL2(), SolutionStairs(), Solution()]

# 1) ví dụ của đề (cách nộp phải hợp lệ; bậc thang còn trùng đúng đáp án mẫu)
assert SolutionStairs().createGrid(2, 3) == ["..#", "#.."]
assert SolutionStairs().createGrid(3, 3) == ["..#", "#..", "##."]
assert SolutionStairs().createGrid(1, 4) == ["...."]
for m, n in [(2, 3), (3, 3), (1, 4), (1, 1), (1, 25), (25, 1), (25, 25), (2, 2), (24, 25)]:
    for b in builders:
        g = b.createGrid(m, n)
        assert valid(g, m, n), (type(b).__name__, m, n)
print("ví dụ và biên OK")

# 2) vét cạn mọi kích thước 1..25 x 1..25
for m in range(1, 26):
    for n in range(1, 26):
        for b in builders:
            assert valid(b.createGrid(m, n), m, n), (type(b).__name__, m, n)
print("kiểm tra toàn miền 25x25 OK")

# 3) ngẫu nhiên có seed + so với vét cạn nhỏ
random.seed(3963)
for _ in range(100):
    m, n = random.randint(1, 4), random.randint(1, 4)
    if m * n <= 12:
        g = SolutionBrute().createGrid(m, n)
        assert valid(g, m, n)
        assert Solution().createGrid(m, n) in all_valid(m, n)
print("đối chiếu với vét cạn OK")

ví dụ và biên OK
kiểm tra toàn miền 25x25 OK


đối chiếu với vét cạn OK


### Test thuộc tính (property test)

(1) Số ô trống của lưới dựng chữ L là `m + n - 1` (đúng bằng độ dài đường đi). (2) **Mọi** lưới có đúng một đường phải có ít nhất `m + n - 1` ô trống. (3) Với lưới bé, đếm số lưới hợp lệ và thấy chữ L nằm trong số đó.

In [16]:
for m in range(1, 26):
    for n in range(1, 26):
        g = Solution().createGrid(m, n)
        assert sum(r.count(".") for r in g) == m + n - 1
for m, n in [(2, 2), (2, 3), (3, 3), (2, 4)]:
    sols = all_valid(m, n)
    assert all(sum(r.count(".") for r in g) >= m + n - 1 for g in sols)
    assert Solution().createGrid(m, n) in sols
    print(f"{m}x{n}: có {len(sols)} lưới hợp lệ")
print("property test OK")

2x2: có 2 lưới hợp lệ
2x3: có 5 lưới hợp lệ
3x3: có 26 lưới hợp lệ
2x4: có 12 lưới hợp lệ
property test OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: để lưới toàn `'.'`

Khi `m, n >= 2`, lưới trống hoàn toàn có `C(m+n-2, m-1)` đường đi (nhiều hơn 1).

In [17]:
def wrong_all_open(m, n):
    return ["." * n for _ in range(m)]                       # SAI: quá nhiều đường

print(count_paths(wrong_all_open(3, 3)), "đường != 1")
assert count_paths(wrong_all_open(3, 3)) == 6 and not valid(wrong_all_open(3, 3), 3, 3)
assert valid(wrong_all_open(1, 5), 1, 5)                      # chỉ đúng khi m = 1 hoặc n = 1

6 đường != 1


### Lỗi 2: chỉ mở hàng đầu (không tới được đích)

Nếu quên mở cột cuối, không có đường nào (`0 != 1`) khi `m >= 2`.

In [18]:
def wrong_top_only(m, n):
    return ["." * n] + ["#" * n for _ in range(m - 1)]       # SAI: không có đường xuống đích

print(count_paths(wrong_top_only(3, 3)), "đường != 1")
assert count_paths(wrong_top_only(3, 3)) == 0 and not valid(wrong_top_only(3, 3), 3, 3)

0 đường != 1


### Lỗi 3: mở cả hàng đầu, hàng cuối và hai cột biên (hình vành khuyên)

Sinh ra **hai** đường (đi trên-rồi-phải, hoặc trái-rồi-dưới).

In [19]:
def wrong_ring(m, n):
    g = [["#"] * n for _ in range(m)]
    for j in range(n):
        g[0][j] = g[m - 1][j] = "."
    for i in range(m):
        g[i][0] = g[i][n - 1] = "."
    return ["".join(r) for r in g]                           # SAI: hai đường

print(count_paths(wrong_ring(3, 3)), "đường != 1")
assert count_paths(wrong_ring(3, 3)) == 2

2 đường != 1


### Lỗi 4: dùng `[["#"] * n] * m` (các hàng cùng tham chiếu)

Mở hàng 0 cũng mở **mọi** hàng (cùng một list), nên lưới toàn `.`.

In [20]:
def wrong_alias(m, n):
    g = [["#"] * n] * m                                      # SAI: m hàng là cùng một list
    g[0][0] = "."
    return ["".join(r) for r in g]

print(wrong_alias(2, 2))
assert wrong_alias(2, 2) == [".#", ".#"] and not valid(wrong_alias(2, 2), 2, 2)

['.#', '.#']


### Lỗi 5: nhầm `m` và `n` (số hàng và số cột)

Đề: `m` hàng, `n` cột. Trả về lưới `n x m` sẽ sai kích thước với `m != n`.

In [21]:
def wrong_swapped(m, n):
    return Solution().createGrid(n, m)                       # SAI: đảo m, n

g = wrong_swapped(2, 3)
print(len(g), len(g[0]), "!= 2 x 3")
assert (len(g), len(g[0])) == (3, 2) and not valid(g, 2, 3)

3 2 != 2 x 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "**hãy xây một cấu trúc thỏa điều kiện** / trả về bất kỳ" -> nghĩ **cấu trúc đơn giản nhất** luôn thỏa (ở đây là chữ L), rồi chứng minh.
- "Có đúng một đường" -> làm cho các ô trống tạo thành **một dải không rẽ nhánh**.
- Luôn tự viết **bộ kiểm tra** (`count_paths` bằng DP): bài có nhiều đáp án, không thể so với một đáp án mẫu.
- Dựng lưới bằng `[["#"] * n for _ in range(m)]` (không dùng `* m` cho list lồng).
- Thử các ca biên: `1 x 1`, `1 x n`, `m x 1`, và ca lớn `25 x 25`.
- Để khớp đúng ví dụ mẫu của đề (nếu muốn), dùng bậc thang; còn để nộp bài thì chữ L ngắn nhất.

## 15. Biến thể của bài

1. **Unique Paths** (62), **Unique Paths II** (63): đếm số đường đi (có/không có chướng ngại).
2. **Minimum Path Sum** (64): đường đi chi phí nhỏ nhất trong lưới.
3. **Number of Islands** (200): làm việc với lưới `'1'/'0'`.
4. Biến thể: tạo lưới có **đúng `k` đường đi** (ví dụ `k = 2` bằng cách mở thêm một ô tạo chu trình hình vuông).
5. Biến thể: lưới có **ít ô trống nhất** nhưng vẫn có đúng một đường (đáp án `m + n - 1`).
6. Biến thể: cho phép đi cả **trái/lên** (lúc đó "đường đi đơn" phức tạp hơn: cần DFS đếm đường đơn).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi lưới | `O(2^(mn) mn)` | `O(mn)` | Chỉ để quan sát nghiệm bé |
| 2. Chữ L trên-phải | hàng 0 + cột cuối | `O(mn)` | `O(mn)` | **Nộp bài**, ngắn nhất |
| 3. Chữ L trái-dưới | cột 0 + hàng cuối | `O(mn)` | `O(mn)` | Chứng tỏ có nhiều nghiệm |
| 4. Bậc thang (mô phỏng) | mở các ô con tốt đi qua | `O(mn)` | `O(mn)` | Khớp đúng ví dụ mẫu |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `SolutionL().createGrid(3, 4)` ra gì? Đường đi duy nhất là gì?

**Bài 2 (tính tay):** dùng DP đếm đường đi cho lưới `["...", ".#.", "..."]`. Có phải lưới hợp lệ của bài này không?

**Bài 3 (điền TODO):** hoàn thành `create_grid_two_paths(m, n)` với `m, n >= 2`: lưới có đúng **2** đường đi.

**Bài 4 (tự viết lại):** viết `path_cells(grid)` trả về danh sách ô trên đường đi duy nhất (nếu lưới hợp lệ).

**Bài 5 (mini challenge):** viết `min_open_cells(m, n)` (trả `m + n - 1`) rồi chứng minh bằng code rằng mọi lưới hợp lệ `3 x 3` có ít nhất chừng đó ô trống.

In [22]:
# Bài 1: đoán rồi chạy
for row in SolutionL().createGrid(3, 4):
    print(row)

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def create_grid_two_paths_todo(m, n):
    # TODO: bắt đầu từ chữ L rồi mở thêm một ô để tạo hai đường
    return []

....
###.
###.


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** Bài 1: hàng 0 toàn `.`; cột cuối `.`. Bài 2: tính `dp` từng ô, lưu ý ô giữa là `#`.

**Mức 2 (vừa):** Bài 3: ở chữ L, mở thêm ô `(1, n-2)` (kề dưới hàng đầu, kề trái cột cuối) tạo thêm một đường chữ nhật nhỏ. Bài 4: bắt đầu `(0,0)`, mỗi bước chọn ô phải hoặc xuống đang là `.` (chỉ có một ô như vậy trừ ô cuối). Bài 5: so với `sum(r.count('.'))` trên `all_valid(3, 3)`.

**Mức 3 (gần đáp án):** Bài 1: `["....", "###.", "###."]`; đường đi `(0,0)->(0,3)->(2,3)`. Bài 2: có 2 đường đi (qua trên hoặc qua dưới ô giữa) nên không hợp lệ. Bài 5: kết quả `5`.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2
assert SolutionL().createGrid(3, 4) == ["....", "###.", "###."]
assert count_paths(["...", ".#.", "..."]) == 2

# Bài 3
def create_grid_two_paths(m, n):
    g = [list(r) for r in SolutionL().createGrid(m, n)]
    g[1][n - 2] = "."
    return ["".join(r) for r in g]

for m, n in [(2, 2), (2, 3), (3, 3), (4, 5), (25, 25)]:
    assert count_paths(create_grid_two_paths(m, n)) == 2, (m, n)

# Bài 4
def path_cells(grid):
    m, n = len(grid), len(grid[0])
    i = j = 0
    cells = [(0, 0)]
    while (i, j) != (m - 1, n - 1):
        if j + 1 < n and grid[i][j + 1] == ".":
            j += 1
        else:
            i += 1
        cells.append((i, j))
    return cells

assert path_cells(["..#", "#.."]) == [(0, 0), (0, 1), (1, 1), (1, 2)]
assert path_cells(["...."]) == [(0, 0), (0, 1), (0, 2), (0, 3)]

# Bài 5
def min_open_cells(m, n):
    return m + n - 1

assert min_open_cells(3, 3) == 5
assert all(sum(r.count(".") for r in g) >= 5 for g in all_valid(3, 3))
assert min(sum(r.count(".") for r in g) for g in all_valid(3, 3)) == 5
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

### Template

```python
class Solution:
    def createGrid(self, m: int, n: int) -> List[str]:
        g = [["#"] * n for _ in range(m)]   # m hàng riêng biệt, toàn chướng ngại
        g[0] = ["."] * n                    # mở hàng đầu
        for i in range(m):
            g[i][n - 1] = "."               # mở cột cuối
        return ["".join(row) for row in g]
```

Khung chung cho bài "xây dựng": chọn **một dải mỏng** nối đầu với cuối, mở đúng dải đó, rồi **kiểm bằng DP**.

### Checklist

- [ ] Đã đọc kỹ constraints (độ dài, miền giá trị) và biết thuật toán chạy được trong giới hạn đó.
- [ ] Đã chạy tay ít nhất **một ví dụ của đề** và một **trường hợp biên** (nhỏ nhất, lớn nhất, giá trị đặc biệt).
- [ ] Đúng **tên class / tên hàm / kiểu trả về** như skeleton LeetCode (đúng `bool`, `int`, `list`...).
- [ ] Không dùng `input()`, không `print` thừa trong code nộp.
- [ ] Không làm hỏng dữ liệu đầu vào nếu đề không cho phép (sao chép trước khi sửa).
- [ ] Đã so sánh với một cách brute force trên input ngẫu nhiên nhỏ.
- [ ] Lưới có đúng `m` chuỗi, mỗi chuỗi dài `n`; chỉ chứa `.` và `#`; ô đầu và ô cuối đều `.`.
- [ ] Đã kiểm tra bằng bộ đếm đường đi (đúng 1) cho `1 x n`, `m x 1`, `m x n`; không dùng `[...] * m` cho list lồng.